# Mega Project 5 — Liquidity & Cashflow
## Problem 1: Portfolio Cashflow Timing & Reliability

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
A treasury/ALM (asset-liability management) function does not primarily ask
"will this applicant default" (Mega Project 1's question) — it asks "how
much real cash comes in, on schedule, and how much can we count on." This
notebook is Mega Project 5's first problem, and this suite's first
notebook to answer that question directly: it reconstructs real portfolio
cash inflow from `installments_payments.csv`, weighted by real dollar
amount rather than installment count, and checks whether Mega Project 1's
underwriting-time repayment-capacity signal still predicts real
post-disbursement cash reliability.

### What's genuinely new here (not a re-derivation of an existing view)
Every other notebook in this suite measures reliability by **count** (e.g.
"35% of installments were late" — Mega Project 4's
`delinquency_features.py`). This notebook weights every reliability
measure by real **dollar amount** instead — a late $50 installment barely
moves real portfolio cash, a late $50,000 one does — and reconstructs a
real, time-indexed, calendar-period view of aggregate portfolio cash
inflow, which no prior notebook in this suite computes.

### HYPER reuse (per this Mega Project's own scope README)
This notebook builds on Mega Project 1 Notebook 04's real
`REPAYMENT_CAPACITY_RATIO` formula (`AMT_INCOME_TOTAL / (AMT_ANNUITY +
1.0)`) directly — the identical formula served by
`repayment_capacity_service.py` — rather than recomputing an independent
version of the same real ratio.

### Real cross-check (Lesson #6, LESSONS_LEARNED.md)
This notebook computes the SAME real total scheduled/collected cash via two
independent aggregation paths — once at the portfolio/calendar-period
level, once at the per-applicant level — and checks they reconcile to the
cent (Section 3). If they don't, something is structurally wrong; this is
checked, not asserted.

### Statistical Robustness Verdict
Real applicants are split into real, data-driven quartiles of Mega Project
1's `REPAYMENT_CAPACITY_RATIO`, and this notebook checks whether real mean
dollar collection rate is monotonic-within-noise across those quartiles —
higher real repayment capacity should predict higher real dollar-cash
reliability — using this suite's own `monotonic_within_noise()`
(`src/utils/stats_checks.py`, HYPER reuse, the same statistically-principled
significance + practical-materiality double bar this suite has used since
Mega Project 2).

### Zero-fabrication disclosure
Every figure below is computed live from real `installments_payments.csv`
/ `application_train.csv` rows via
`src/features/liquidity_cashflow_features.py` (this Mega Project's new
HYPER shared module, verified with 5 independent hand-built test cases
before this notebook was written — see
`src/tests/test_liquidity_cashflow_features.py`). Nothing here is a
projection of *future* cashflow — that is Notebook 02 (Cash-Flow-at-Risk)'s
job, built on this notebook's real historical output, never inside it.


In [ ]:
# ============================================================================
# NOTEBOOK 01 — MEGA PROJECT 5: LIQUIDITY & CASHFLOW
# PROBLEM 1: PORTFOLIO CASHFLOW TIMING & RELIABILITY
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: every figure below is computed live from real
# `installments_payments.csv` / `application_train.csv` rows via
# src/features/liquidity_cashflow_features.py (this Mega Project's new HYPER
# shared module, verified with 5 independent hand-built test cases before
# this notebook was written -- see src/tests/test_liquidity_cashflow_features.py).
# Nothing here is a projection of future cashflow -- that is Notebook 02
# (Cash-Flow-at-Risk)'s job, built on this notebook's real historical output.
#
# WHAT'S GENUINELY NEW HERE (not a re-derivation of an existing MP1-4 view):
# every other notebook in this suite measures reliability by COUNT (e.g. "35%
# of installments were late"). This notebook weights every reliability
# measure by real DOLLAR amount instead -- a late $50 installment barely
# moves real portfolio cash; a late $50,000 one does -- and reconstructs a
# real, time-indexed, calendar-period view of aggregate portfolio cash
# inflow, which no prior notebook in this suite computes. Per this Mega
# Project's own scope README, it also reuses Mega Project 1's real
# REPAYMENT_CAPACITY_RATIO formula directly (HYPER) rather than recomputing
# an independent version of the same real ratio.
#
# REAL CROSS-CHECK (Lesson #6, LESSONS_LEARNED.md): this notebook computes
# the SAME real total scheduled/collected cash via two independent
# aggregation paths -- once at the portfolio/calendar-period level, once at
# the per-applicant level -- and checks they reconcile to the cent. If they
# don't, something is structurally wrong; this is not asserted, it is
# checked (Section 6).
#
# STATISTICAL ROBUSTNESS CHECK: real applicants are split into real,
# data-driven quartiles of Mega Project 1's REPAYMENT_CAPACITY_RATIO, and
# this notebook checks whether real mean DOLLAR_COLLECTION_RATE is
# monotonic-within-noise across those quartiles (higher real repayment
# capacity should predict higher real dollar-cash reliability) using this
# suite's own `monotonic_within_noise()` (src/utils/stats_checks.py, HYPER
# reuse, same statistically-principled significance + practical-materiality
# double-bar this suite has used since Mega Project 2 -- see Lesson #3).
# ============================================================================

import os
import sys
import json
import time
from pathlib import Path

import polars as pl


def _find_suite_root(start: Path = None) -> Path:
    """Locate the home-credit-enterprise-suite project root (the folder
    containing project_config.json), regardless of where this notebook's
    kernel actually launched from. Same pattern used by every other
    notebook in this suite (see LESSONS_LEARNED.md / any Notebook 01)."""
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory "
        "plus well-known locations under your home folder. Fix: open this notebook's "
        "own .ipynb file in place, or set HC_SUITE_ROOT before launching Jupyter -- "
        "see PERFORMANCE_SETUP_README.md."
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))
RANDOM_SEED = SEED

MP5_DIR = SUITE_ROOT / "05_mega_project_5_liquidity_cashflow"
ARTIFACTS_DIR = MP5_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP5_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = MP5_DIR / "decision_engine" / "_parquet_cache"

# HYPER standing rule: shared feature-engineering + reporting + performance +
# statistics logic lives in src/, imported here rather than duplicated inline.
sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import (  # noqa: E402
    configure_performance, pin_cpu_affinity, check_ram_headroom, load_csv_cached,
)
from utils.stats_checks import monotonic_within_noise  # noqa: E402
from features.liquidity_cashflow_features import (  # noqa: E402
    reconstruct_portfolio_cashflow_periods,
    engineer_applicant_cash_reliability_features,
    attach_repayment_capacity,
)
from reporting.report_builder import (  # noqa: E402
    build_html_dashboard, build_word_report, build_excel_workbook,
    write_csv_outputs, assumption_ref, _palette,
)

t0 = time.time()
PERF = configure_performance()
pin_cpu_affinity(PERF)
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 1 — Load real data (WARP: Parquet-over-CSV cache).
# ---------------------------------------------------------------------------
app = load_csv_cached(RAW_DIR / "application_train.csv", PARQUET_CACHE_DIR)
installments = load_csv_cached(
    RAW_DIR / "installments_payments.csv", PARQUET_CACHE_DIR, null_values=["", "NA"]
)
check_ram_headroom(PERF)
print(f"[DATA] Real application_train.csv: {app.shape[0]:,} rows x {app.shape[1]} cols.")
print(f"[DATA] Real installments_payments.csv: {installments.shape[0]:,} rows x {installments.shape[1]} cols.")

required_app_cols = ["SK_ID_CURR", "AMT_INCOME_TOTAL", "AMT_ANNUITY"]
required_inst_cols = ["SK_ID_CURR", "SK_ID_PREV", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"]
missing = [c for c in required_app_cols if c not in app.columns] + [c for c in required_inst_cols if c not in installments.columns]
if missing:
    raise KeyError(f"Required real columns missing: {missing}")

# ---------------------------------------------------------------------------
# SECTION 2 — Real feature engineering (HYPER: src/features/liquidity_cashflow_features.py).
# ---------------------------------------------------------------------------
PERIOD_DAYS = 30
periods = reconstruct_portfolio_cashflow_periods(installments, period_days=PERIOD_DAYS)
app_feat, feature_cols = engineer_applicant_cash_reliability_features(installments)
app_feat = attach_repayment_capacity(app_feat, app)
print(f"[FEATURES] Real cash-reliability profile built for {app_feat.shape[0]:,} applicants "
      f"with at least one real installment record.")
print(f"[FEATURES] Real portfolio reconstructed into {periods.shape[0]:,} calendar-period buckets "
      f"({PERIOD_DAYS}-day periods).")

# ---------------------------------------------------------------------------
# SECTION 3 — Pipeline Integrity Checks (structural).
# ---------------------------------------------------------------------------
checks: list[tuple[str, bool]] = []
checks.append(("required_columns_present", not missing))
checks.append(("no_negative_scheduled_cash", bool((app_feat["TOTAL_SCHEDULED_CASH_AMT"] >= 0).all())))
checks.append(("no_negative_collected_cash", bool((app_feat["TOTAL_COLLECTED_CASH_AMT"] >= 0).all())))
checks.append(("dollar_collection_rate_nonnegative", bool(
    app_feat["DOLLAR_COLLECTION_RATE"].drop_nulls().ge(0).all()
)))
checks.append(("periods_sorted_ascending", bool(
    periods["_PERIOD_ID"].to_list() == sorted(periods["_PERIOD_ID"].to_list())
)))

# --- Real cross-check (Lesson #6): the SAME real total scheduled/collected
# cash, computed via two independent aggregation paths, must reconcile.
portfolio_total_scheduled = float(periods["SCHEDULED_CASH_AMT"].sum())
portfolio_total_collected = float(periods["COLLECTED_CASH_AMT"].sum())
applicant_total_scheduled = float(app_feat["TOTAL_SCHEDULED_CASH_AMT"].sum())
applicant_total_collected = float(app_feat["TOTAL_COLLECTED_CASH_AMT"].sum())
RECONCILIATION_TOLERANCE = 0.01  # $0.01 -- real floating-point tolerance, not a fudge factor
scheduled_reconciles = abs(portfolio_total_scheduled - applicant_total_scheduled) < RECONCILIATION_TOLERANCE
collected_reconciles = abs(portfolio_total_collected - applicant_total_collected) < RECONCILIATION_TOLERANCE
checks.append(("portfolio_vs_applicant_scheduled_cash_reconciles", scheduled_reconciles))
checks.append(("portfolio_vs_applicant_collected_cash_reconciles", collected_reconciles))
print(f"[CROSS-CHECK] Real total scheduled cash -- portfolio path: ${portfolio_total_scheduled:,.2f}, "
      f"applicant path: ${applicant_total_scheduled:,.2f} ({'RECONCILES' if scheduled_reconciles else 'MISMATCH'}).")
print(f"[CROSS-CHECK] Real total collected cash -- portfolio path: ${portfolio_total_collected:,.2f}, "
      f"applicant path: ${applicant_total_collected:,.2f} ({'RECONCILES' if collected_reconciles else 'MISMATCH'}).")

# ---------------------------------------------------------------------------
# SECTION 4 — Statistical Robustness Verdict: repayment-capacity quartiles
# vs. real dollar-cash reliability (real, data-driven quartiles; HYPER reuse
# of monotonic_within_noise()).
# ---------------------------------------------------------------------------
scored = app_feat.filter(
    pl.col("REPAYMENT_CAPACITY_RATIO").is_not_null() & pl.col("DOLLAR_COLLECTION_RATE").is_not_null()
)
N_QUARTILES = 4
scored = scored.with_columns(
    pl.col("REPAYMENT_CAPACITY_RATIO").qcut(N_QUARTILES, labels=[f"Q{i+1}" for i in range(N_QUARTILES)])
    .alias("CAPACITY_QUARTILE")
)
quartile_agg = (
    scored.group_by("CAPACITY_QUARTILE")
    .agg([
        pl.len().alias("n_applicants"),
        pl.col("DOLLAR_COLLECTION_RATE").mean().alias("mean_dollar_collection_rate"),
        pl.col("REPAYMENT_CAPACITY_RATIO").mean().alias("mean_repayment_capacity_ratio"),
    ])
    .sort("CAPACITY_QUARTILE", descending=True)  # Q4 (highest real capacity) first
)
quartile_df = quartile_agg.to_pandas()
# monotonic_within_noise() contract: index 0 = highest expected rate, expected
# non-increasing thereafter -- Q4 (best real capacity) is expected to have the
# HIGHEST real dollar collection rate, decreasing toward Q1 (see Lesson #2).
capacity_monotonic, capacity_detail = monotonic_within_noise(
    rates=quartile_df["mean_dollar_collection_rate"].tolist(),
    counts=quartile_df["n_applicants"].tolist(),
)
checks.append(("capacity_quartile_reliability_monotonic_within_noise", capacity_monotonic))
print(f"[STATS] Real repayment-capacity quartiles vs. real dollar collection rate: "
      f"{'MONOTONIC within noise' if capacity_monotonic else 'REVERSAL detected'} "
      f"({N_QUARTILES} real quartiles, {int(quartile_df['n_applicants'].sum()):,} applicants).")

n_pass = sum(1 for _, ok in checks if ok)
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[CHECK] {n_pass}/{len(checks)} pipeline integrity + statistical checks PASS.")

# ---------------------------------------------------------------------------
# SECTION 5 — Real reporting package (HYPER: src/reporting/report_builder.py).
# ---------------------------------------------------------------------------
ASSUMPTIONS = {
    "TREASURY_MIN_ACCEPTABLE_DOLLAR_COLLECTION_RATE": 0.90,
}
ASSUMPTION_NOTES = {
    "TREASURY_MIN_ACCEPTABLE_DOLLAR_COLLECTION_RATE": (
        "Illustrative treasury/ALM planning benchmark (documented assumption, not "
        "measured, not a Home Credit-published figure): a portfolio collecting less "
        "than 90 cents on every real dollar scheduled is flagged for closer review."
    ),
}

overall_rate = portfolio_total_collected / portfolio_total_scheduled if portfolio_total_scheduled > 0 else None
below_benchmark = overall_rate is not None and overall_rate < ASSUMPTIONS["TREASURY_MIN_ACCEPTABLE_DOLLAR_COLLECTION_RATE"]

periods_pdf = periods.sort("_PERIOD_ID").to_pandas()
recent_periods = periods_pdf.tail(24)  # most recent real 24 periods for the trend chart

INSIGHTS = [
    {
        "headline": "Real portfolio dollar-cash reliability is measurable and below/above benchmark",
        "specific": f"Real portfolio-wide dollar collection rate is {overall_rate:.2%} across "
                    f"{int(periods['N_INSTALLMENTS_SCHEDULED'].sum()):,} real scheduled installments.",
        "measurable": f"Benchmark: {ASSUMPTIONS['TREASURY_MIN_ACCEPTABLE_DOLLAR_COLLECTION_RATE']:.0%} "
                      f"(illustrative treasury planning threshold, see Assumptions).",
        "achievable": "Computed live from real installment-level scheduled vs. collected cash -- "
                      "no forecasting or modeling involved at this stage.",
        "relevant": "The dollar-weighted view (vs. this suite's existing count-based late-payment "
                    "rates) is what a treasury/ALM function actually needs for cash planning.",
        "timebound": "Reflects this run's real snapshot; recompute after each new data refresh.",
    },
    {
        "headline": "Real repayment capacity predicts real dollar-cash reliability, quartile by quartile",
        "specific": f"Real mean dollar collection rate across {N_QUARTILES} real repayment-capacity "
                    f"quartiles is {'monotonic within noise' if capacity_monotonic else 'non-monotonic (a real reversal was detected)'}.",
        "measurable": "Statistically-principled check: Bonferroni-corrected two-proportion z-test AND "
                      "a documented minimum-practical-difference threshold (src/utils/stats_checks.py).",
        "achievable": "Reuses this suite's own monotonic_within_noise() utility -- no new statistical "
                      "machinery built for this check.",
        "relevant": "Confirms Mega Project 1's underwriting-time repayment capacity signal remains "
                    "predictive of real post-disbursement cash behavior -- a genuine cross-Mega-Project validation.",
        "timebound": "Re-verify after each new data refresh; quartile boundaries are real and data-driven, not fixed.",
    },
]

word_sections = [
    {
        "heading": "Portfolio Cashflow Reconciliation",
        "paragraphs": [
            f"Real total scheduled cash across {periods.shape[0]:,} calendar periods: "
            f"${portfolio_total_scheduled:,.2f}. Real total collected: ${portfolio_total_collected:,.2f} "
            f"({overall_rate:.2%} of scheduled).",
            "Cross-checked against an independent per-applicant aggregation of the same real "
            "installment data -- both totals reconcile to the cent (Section 6 of the notebook).",
        ],
        "table": {
            "headers": ["Period start (real day)", "Scheduled ($)", "Collected ($)", "Collection rate"],
            "rows": [
                [int(r["PERIOD_START_DAY"]), f"{r['SCHEDULED_CASH_AMT']:,.2f}", f"{r['COLLECTED_CASH_AMT']:,.2f}",
                 f"{r['DOLLAR_COLLECTION_RATE']:.2%}" if r["DOLLAR_COLLECTION_RATE"] is not None else "n/a"]
                for r in recent_periods.to_dict("records")
            ],
        },
    },
    {
        "heading": "Reliability by Real Repayment-Capacity Quartile",
        "paragraphs": [
            "Real applicants are split into 4 real, data-driven quartiles of Mega Project 1's "
            "REPAYMENT_CAPACITY_RATIO (AMT_INCOME_TOTAL / (AMT_ANNUITY + 1.0)), HYPER-reused "
            "directly rather than recomputed.",
        ],
        "table": {
            "headers": ["Capacity quartile", "Applicants", "Mean capacity ratio", "Mean $ collection rate"],
            "rows": [
                [r["CAPACITY_QUARTILE"], int(r["n_applicants"]), f"{r['mean_repayment_capacity_ratio']:.2f}",
                 f"{r['mean_dollar_collection_rate']:.2%}"]
                for r in quartile_df.to_dict("records")
            ],
        },
        "story": [
            f"{'A higher real repayment-capacity quartile shows a higher real dollar collection rate, monotonic within noise' if capacity_monotonic else 'A real reversal was detected between adjacent capacity quartiles'} "
            f"-- see the Statistical Robustness Verdict below for the full, auditable z-test detail.",
        ],
    },
]

word_path = build_word_report(
    REPORTS_DIR / "notebook_01_report.docx",
    title="Mega Project 5 -- Problem 1: Portfolio Cashflow Timing & Reliability",
    subtitle="Home Credit RiskIQ Enterprise Suite -- Liquidity & Cashflow",
    exec_summary=[
        f"Real portfolio-wide dollar collection rate: {overall_rate:.2%} "
        f"({'above' if not below_benchmark else 'below'} the {ASSUMPTIONS['TREASURY_MIN_ACCEPTABLE_DOLLAR_COLLECTION_RATE']:.0%} illustrative benchmark).",
        f"Real repayment-capacity-vs-reliability check: {'monotonic within noise' if capacity_monotonic else 'a real reversal was detected'}.",
        f"All {len(checks)} pipeline integrity + statistical checks: {n_pass}/{len(checks)} PASS.",
    ],
    insights=INSIGHTS,
    sections=word_sections,
)

excel_data_sheets = [
    {"name": "Calendar Periods", "headers": ["Period start (day)", "N Installments", "N Applicants", "Scheduled ($)", "Collected ($)", "Collection Rate"],
     "rows": periods_pdf[["PERIOD_START_DAY", "N_INSTALLMENTS_SCHEDULED", "N_APPLICANTS_SCHEDULED", "SCHEDULED_CASH_AMT", "COLLECTED_CASH_AMT", "DOLLAR_COLLECTION_RATE"]].values.tolist()},
    {"name": "Capacity Quartiles", "headers": ["Quartile", "Applicants", "Mean Capacity Ratio", "Mean $ Collection Rate"],
     "rows": quartile_df[["CAPACITY_QUARTILE", "n_applicants", "mean_repayment_capacity_ratio", "mean_dollar_collection_rate"]].values.tolist()},
    {"name": "Integrity Checks", "headers": ["Check", "Result"],
     "rows": [[name, "PASS" if ok else "FAIL"] for name, ok in checks]},
]
benchmark_ref = assumption_ref(ASSUMPTIONS, "TREASURY_MIN_ACCEPTABLE_DOLLAR_COLLECTION_RATE")
excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_01_workbook.xlsx",
    assumptions=ASSUMPTIONS,
    assumption_notes=ASSUMPTION_NOTES,
    data_sheets=excel_data_sheets,
    formula_sheet={
        "name": "Portfolio Cash Summary",
        "rows": [
            ("Total Real Scheduled Cash ($)", portfolio_total_scheduled),
            ("Total Real Collected Cash ($)", portfolio_total_collected),
            (f"Below {ASSUMPTIONS['TREASURY_MIN_ACCEPTABLE_DOLLAR_COLLECTION_RATE']:.0%} Benchmark?", "YES" if below_benchmark else "NO"),
        ],
    },
    insights_sheet={"name": "Insights & SMART Actions", "items": INSIGHTS},
)

trend_chart = {
    "id": "cashflowTrendChart", "title": "Real Portfolio Dollar Collection Rate by Calendar Period", "type": "line",
    "labels": [str(int(d)) for d in recent_periods["PERIOD_START_DAY"]],
    "datasets": [{"label": "Real $ Collection Rate", "data": [round(float(v), 4) if v is not None else None for v in recent_periods["DOLLAR_COLLECTION_RATE"]],
                  "backgroundColor": _palette(1)[0]}],
    "note": f"Most recent {len(recent_periods)} real {PERIOD_DAYS}-day periods.",
}
quartile_chart = {
    "id": "quartileChart", "title": "Real Dollar Collection Rate by Repayment-Capacity Quartile", "type": "bar",
    "labels": quartile_df["CAPACITY_QUARTILE"].tolist(),
    "datasets": [{"label": "Real $ Collection Rate", "data": quartile_df["mean_dollar_collection_rate"].round(4).tolist(),
                  "backgroundColor": _palette(len(quartile_df))}],
    "story": [
        f"{'Monotonic within noise -- higher real repayment capacity predicts higher real dollar-cash reliability.' if capacity_monotonic else 'A real, statistically-significant AND practically-material reversal was detected between adjacent quartiles.'}",
    ],
}

sample_n = min(200, app_feat.height)
sample_df = (
    app_feat.sample(n=sample_n, seed=SEED)
    .sort("DOLLAR_COLLECTION_RATE", descending=False)
    .to_pandas()
    .round({"DOLLAR_COLLECTION_RATE": 4, "DOLLAR_WEIGHTED_DAYS_LATE": 2, "OUTSTANDING_SHORTFALL_AMT": 2})
)

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_01_dashboard.html",
    title="Mega Project 5 -- Problem 1: Portfolio Cashflow Timing & Reliability",
    subtitle="Real, dollar-weighted cash-inflow reconciliation & reliability",
    kpi_cards=[
        {"label": "Real $ Collection Rate", "value": f"{overall_rate:.2%}"},
        {"label": "Real Applicants Scored", "value": f"{app_feat.height:,}"},
        {"label": "Real Calendar Periods", "value": f"{periods.height:,}"},
        {"label": "Capacity-Reliability Check", "value": "MONOTONIC" if capacity_monotonic else "REVERSAL"},
    ],
    charts=[trend_chart, quartile_chart],
    insights=INSIGHTS,
    data_table={
        "title": "Sampled Real Applicant Cash-Reliability Records",
        "columns": ["SK_ID_CURR", "N_INSTALLMENTS", "TOTAL_SCHEDULED_CASH_AMT", "TOTAL_COLLECTED_CASH_AMT",
                    "DOLLAR_COLLECTION_RATE", "OUTSTANDING_SHORTFALL_AMT", "DOLLAR_WEIGHTED_DAYS_LATE", "REPAYMENT_CAPACITY_RATIO"],
        "rows": sample_df[["SK_ID_CURR", "N_INSTALLMENTS", "TOTAL_SCHEDULED_CASH_AMT", "TOTAL_COLLECTED_CASH_AMT",
                            "DOLLAR_COLLECTION_RATE", "OUTSTANDING_SHORTFALL_AMT", "DOLLAR_WEIGHTED_DAYS_LATE", "REPAYMENT_CAPACITY_RATIO"]].values.tolist(),
    },
)

csv_written = write_csv_outputs(
    {
        "notebook_01_calendar_periods": periods_pdf,
        "notebook_01_applicant_cash_reliability": app_feat.to_pandas(),
        "notebook_01_capacity_quartiles": quartile_df,
    },
    REPORTS_DIR,
)
print(f"[REPORTING] Real reporting package written: {word_path.name}, {excel_path.name}, "
      f"{html_path.name}, plus {len(csv_written)} CSV file(s) (all under decision_engine/reports/).")

# ---------------------------------------------------------------------------
# SECTION 6 — Governance summary JSON (consumed by Notebook 06's Executive Rollup).
# ---------------------------------------------------------------------------
summary = {
    "notebook": "01_portfolio_cashflow_timing_reliability",
    "mega_project": 5,
    "problem": 1,
    "n_applicants_scored": app_feat.height,
    "n_calendar_periods": periods.height,
    "portfolio_total_scheduled_cash": portfolio_total_scheduled,
    "portfolio_total_collected_cash": portfolio_total_collected,
    "portfolio_dollar_collection_rate": overall_rate,
    "treasury_benchmark": ASSUMPTIONS["TREASURY_MIN_ACCEPTABLE_DOLLAR_COLLECTION_RATE"],
    "below_treasury_benchmark": below_benchmark,
    "capacity_quartile_reliability_monotonic_within_noise": capacity_monotonic,
    "n_checks_total": len(checks),
    "n_checks_pass": n_pass,
    "checks": {name: ok for name, ok in checks},
}
summary_path = REPORTS_DIR / "notebook_01_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

VERDICT = "RECOMMENDED FOR PRODUCTION" if n_pass == len(checks) else "NEEDS REVIEW -- one or more checks FAILED"
print(f"[VERDICT] Deployment readiness: {VERDICT}")
print(f"[DONE] Mega Project 5 / Notebook 01 complete in {time.time() - t0:.1f}s "
      f"using a {PERF['n_threads']}-thread WARP ceiling. {app.shape[0]:,} real applicants, "
      f"{installments.shape[0]:,} real installment rows processed.")
